# Backfill Period Metrics

Use this notebook to manually add TEST and IS Aggregate Data metrics to existing raw BO log rows.

This is a maintenance workflow only. It does not rank trials, plot results, or run analysis.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from backfill_period_metrics import (
    load_log_csv,
    parse_aggregate_data_block,
    row_display_columns,
    rows_missing_period_metrics,
    save_log_csv,
    update_row_period_metrics,
    update_row_period_metrics_from_block,
)

## Choose Raw Log

In [ ]:
LOG_CSV = "brain_bo_usa_top3000_angze.csv"

df = load_log_csv(LOG_CSV)
print(f"Loaded {len(df)} row(s) from {LOG_CSV}")

## Find Rows Missing TEST Or IS Metrics

In [ ]:
missing_test = rows_missing_period_metrics(df, period="test")
missing_is = rows_missing_period_metrics(df, period="is")

print(f"Rows missing TEST metrics: {len(missing_test)}")
print(f"Rows missing IS metrics:   {len(missing_is)}")

missing_test[row_display_columns(missing_test)].tail(10)

## Select Row To Backfill

Set `ROW_INDEX` to the DataFrame index of the row you want to update.

In [ ]:
ROW_INDEX = 58

selected = df.loc[[ROW_INDEX], row_display_columns(df)]
selected

## Paste TEST and IS Metrics

Paste the TEST and IS Aggregate Data block between the triple quotes below. Leave it empty to skip. This avoids notebook `input()` issues with multi-line paste.

In [ ]:
TEST_BLOCK = """

Sharpe
0.18
Turnover
29.87%
Fitness
0.03
Returns
0.99%
Drawdown
5.97%
Margin
0.66‱
"""

if TEST_BLOCK.strip():
    df = update_row_period_metrics_from_block(df, ROW_INDEX, "test", TEST_BLOCK)
    print("Updated TEST metrics.")
else:
    print("Skipped TEST metrics.")

IS_BLOCK = """
Sharpe
-0.15
Turnover
30.14%
Fitness
-0.03
Returns
-1.07%
Drawdown
22.75%
Margin
-0.71‱
"""

if IS_BLOCK.strip():
    df = update_row_period_metrics_from_block(df, ROW_INDEX, "is", IS_BLOCK)
    print("Updated IS metrics.")
else:
    print("Skipped IS metrics.")

df.loc[[ROW_INDEX], row_display_columns(df)]

## Save Updated CSV

Saving creates a timestamped backup first.

In [ ]:
backup_path = save_log_csv(df, LOG_CSV, backup=False)
print(f"Saved updated log to: {LOG_CSV}")
print(f"Backup saved to: {backup_path}")

Reminder: this notebook only backfills period metrics. Do analysis in a separate analysis notebook.